**Question 1**

In [13]:
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib3.util import Retry
from requests.adapters import HTTPAdapter

# Setup variables
roll_no = "23L_0573"
domain = "https://sandbox.oxylabs.io"
url = f"{domain}/products"

# Setup session with retry limits
req_session = requests.Session()
retry_strategy = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[500, 502, 503, 504]
)
req_session.mount('https://', HTTPAdapter(max_retries=retry_strategy))
req_session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
})

data = []
seen_links = set()
page_cnt = 1

print("Starting scraper process...")

while url:
    print(f"Fetching page {page_cnt}: {url}")

    res = req_session.get(url)
    page_soup = BeautifulSoup(res.text, 'html.parser')

    cards = page_soup.select('.product-card')
    print(f"Found {len(cards)} items on page {page_cnt}")

    for c in cards:
        t_tag = c.select_one('.title')
        p_tag = c.select_one('.price-wrapper')
        l_tag = c.select_one('a.card-header[href]')

        if not l_tag or not l_tag.get('href'):
            continue

        rel_path = l_tag['href']
        full_link = rel_path if rel_path.startswith('http') else domain + rel_path

        # Skip if already processed
        if full_link in seen_links:
            continue
        seen_links.add(full_link)

        # Fetch individual item details
        try:
            time.sleep(0.5)
            d_res = req_session.get(full_link, timeout=10)
            d_soup = BeautifulSoup(d_res.text, 'html.parser')

            s_tag = d_soup.select_one('.availability')
            desc_tag = d_soup.select_one('.description')

            stock_txt = s_tag.get_text(strip=True) if s_tag else "Not found"
            desc_txt = desc_tag.get_text(strip=True) if desc_tag else ""
        except Exception:
            stock_txt = "Error loading"
            desc_txt = ""

        data.append({
            'Product name': t_tag.get_text(strip=True) if t_tag else None,
            'Price': p_tag.get_text(strip=True) if p_tag else None,
            'Stock status': stock_txt,
            'Description': desc_txt,
            'Detail URL': full_link,
            'Listing Page Number': page_cnt
        })

    # Check next page
    next_pg = page_soup.select_one('a[aria-label*="next i"], a[rel="next"], .next-page, a.next')

    if next_pg and next_pg.get('href'):
        n_href = next_pg['href']
        url = n_href if n_href.startswith('http') else domain + n_href
        page_cnt += 1
    else:
        print("Done. No more pages found.")
        url = None

# Export results
results_df = pd.DataFrame(data)
results_df.drop_duplicates(subset=['Detail URL'], inplace=True)

out_file = f"{roll_no}_version_static_products.csv"
results_df.to_csv(out_file, index=False)

print("\nTask finished successfully.")
print(f"Pages checked: {page_cnt}")
print(f"Unique entries saved: {len(results_df)}")
print(f"Output path: {out_file}")

Starting Question 1 Scraper...

Fetching Listing Page 1: https://sandbox.oxylabs.io/products
Found 32 products on page 1.

Fetching Listing Page 2: https://sandbox.oxylabs.io/products?page=2
Found 32 products on page 2.

Fetching Listing Page 3: https://sandbox.oxylabs.io/products?page=3
Found 32 products on page 3.

Fetching Listing Page 4: https://sandbox.oxylabs.io/products?page=4
Found 32 products on page 4.

Fetching Listing Page 5: https://sandbox.oxylabs.io/products?page=5
Found 32 products on page 5.

Fetching Listing Page 6: https://sandbox.oxylabs.io/products?page=6
Found 32 products on page 6.

Fetching Listing Page 7: https://sandbox.oxylabs.io/products?page=7
Found 32 products on page 7.

Fetching Listing Page 8: https://sandbox.oxylabs.io/products?page=8
Found 32 products on page 8.

Fetching Listing Page 9: https://sandbox.oxylabs.io/products?page=9
Found 32 products on page 9.

Fetching Listing Page 10: https://sandbox.oxylabs.io/products?page=10
Found 32 products on pa

**QUESTION 2**

In [5]:

!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb || apt-get -f install -y
!pip install selenium webdriver-manager beautifulsoup4 pandas

--2026-09-04 16:05:05--  https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
Resolving dl.google.com (dl.google.com)... 142.251.111.91, 142.251.111.190, 142.251.111.93, ...
Connecting to dl.google.com (dl.google.com)|142.251.111.91|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 141157492 (135M) [application/x-debian-package]
Saving to: ‘google-chrome-stable_current_amd64.deb’

google-chrome-stabl 100%[===================>] 134.62M   229MB/s    in 0.6s    

2026-09-04 16:05:06 (229 MB/s) - ‘google-chrome-stable_current_amd64.deb’ saved [141157492/141157492]

Selecting previously unselected package google-chrome-stable.
(Reading database ... 118885 files and directories currently installed.)
Preparing to unpack google-chrome-stable_current_amd64.deb ...
Unpacking google-chrome-stable (152.0.7977.82-1) ...
dpkg: dependency problems prevent configuration of google-chrome-stable:
 google-chrome-stable depends on libatk-bridge2.0-0 (>= 2.5.3);

In [11]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# Chrome driver setup
chrome_opts = webdriver.ChromeOptions()
chrome_opts.add_argument("--headless")
chrome_opts.add_argument("--no-sandbox")
chrome_opts.add_argument("--disable-dev-shm-usage")
chrome_opts.binary_location = "/usr/bin/google-chrome"

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_opts)

# Infinite scroll scraping
target_url = "https://www.scrapingcourse.com/infinite-scrolling"
driver.get(target_url)
time.sleep(3)

items_list = []
seen_urls = set()
batch = 0

last_h = driver.execute_script("return document.body.scrollHeight")
print("Doc height =", last_h)

while True:
    cards = driver.find_elements(By.CSS_SELECTOR, ".product-item")

    for card in cards:
        try:
            link = card.find_element(By.TAG_NAME, "a").get_attribute("href")

            if link and link not in seen_urls:
                seen_urls.add(link)

                # Extract basic card details
                try:
                    name = card.find_element(By.CSS_SELECTOR, ".product-name").text.strip()
                except Exception:
                    name = "N/A"

                try:
                    price = card.find_element(By.CSS_SELECTOR, ".product-price").text.strip()
                except Exception:
                    price = "N/A"

                try:
                    img = card.find_element(By.TAG_NAME, "img").get_attribute("src")
                except Exception:
                    img = "N/A"

                items_list.append({
                    "Product name": name,
                    "Price": price,
                    "Image URL": img,
                    "Scroll batch": batch,
                    "Product URL": link
                })
        except Exception:
            continue

    # Scroll down to load more items
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)

    new_h = driver.execute_script("return document.body.scrollHeight")
    if new_h == last_h:
        break

    last_h = new_h
    batch += 1

driver.quit()

# Fetch detail pages via BeautifulSoup
req_session = requests.Session()
req_session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"})

final_data = []

for item in items_list:
    p_url = item["Product URL"]
    sku_val = "N/A"
    desc_val = "N/A"

    try:
        r = req_session.get(p_url, timeout=10)
        if r.status_code == 200:
            soup = BeautifulSoup(r.content, "html.parser")

            sku_node = soup.select_one('.sku')
            if sku_node:
                sku_val = sku_node.get_text(strip=True)

            desc_node = soup.select_one('.woocommerce-product-details__short-description')
            if desc_node:
                desc_val = desc_node.get_text(strip=True)
    except Exception:
        pass

    final_data.append({
        "Product name": item["Product name"],
        "Price": item["Price"],
        "Image URL": item["Image URL"],
        "Scroll batch": item["Scroll batch"],
        "SKU": sku_val,
        "Short description": desc_val,
        "Product URL": p_url
    })

# Output to CSV
df = pd.DataFrame(final_data)
out_csv = "23L_0573_versionA_dynamic_products.csv"
df.to_csv(out_csv, index=False)

print(f"Scrape complete! Saved {len(df)} products over {batch + 1} scroll batches.")

Doc height = 2648
Scrape complete! Saved 147 products over 16 scroll batches.
